# 1导入库配置环境

In [1]:
!pip install darts  # 如果未安装darts，可以取消注释
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print("警告已忽略！")
from darts import TimeSeries
from darts.dataprocessing import Pipeline
from darts.dataprocessing.transformers import Scaler, InvertibleMapper, StaticCovariatesTransformer
from darts.dataprocessing.transformers.missing_values_filler import MissingValuesFiller
from darts.metrics import rmsle
from darts.models import LinearRegressionModel, LightGBMModel, XGBModel, CatBoostModel
from darts.models.filtering.moving_average_filter import MovingAverageFilter
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from tqdm.notebook import tqdm_notebook

plt.style.use("ggplot")
plt.rcParams["font.size"] = 15
COLORS = list(sns.color_palette())

# 辅助函数打印消息
def cprint(title, *args):
    print(
        "="*len(title), title, "="*len(title),
        *args,
        sep="\n",
    )

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.3/46.3 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.7/204.7 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.4/825.4 kB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 4.1 MB/s eta 0:00:00
  Attempting uninstall: pytorch-lightning
    Found existing installation: pytorch-lightning 2.6.0
    Uninstalling pytorch-lightning-2.6.0:
      Successfully uninstalled pytorch-lightning-2.6.0
警告已忽略！


The StatsForecast module could not be imported. To enable support for the AutoARIMA, AutoETS and Croston models, please consider installing it.


# 2载入数据查看数据基本信息

In [2]:
PATH = "/kaggle/input/store-sales-time-series-forecasting"

train = pd.read_csv(os.path.join(PATH, "train.csv"), parse_dates=["date"])
test = pd.read_csv(os.path.join(PATH, "test.csv"), parse_dates=["date"])

oil = pd.read_csv(os.path.join(PATH, "oil.csv"), parse_dates=["date"]).rename(columns={"dcoilwtico": "oil"})
store = pd.read_csv(os.path.join(PATH, "stores.csv"))
transaction = pd.read_csv(os.path.join(PATH, "transactions.csv"), parse_dates=["date"])
holiday = pd.read_csv(os.path.join(PATH, "holidays_events.csv"), parse_dates=["date"])

# 检查：训练数据中存在缺失的日期
num_family = train.family.nunique()
num_store = train.store_nbr.nunique()
num_ts = train.groupby(["store_nbr", "family"]).ngroups
train_start = train.date.min().date()
train_end = train.date.max().date()
num_train_date = train.date.nunique()
train_len = (train_end - train_start).days + 1
test_start = test.date.min().date()
test_end = test.date.max().date()
num_test_date = test.date.nunique()
test_len = (test_end - test_start).days + 1

cprint(
    "数据基本信息",
    f"产品系列类型数量      : {num_family}",
    f"门店数量            : {num_store}",
    f"门店-产品系列组合数量: {num_ts}",
    f"目标序列数量        : {num_ts}",
    "",
    f"训练日期唯一值数量: {num_train_date}",
    f"训练日期范围      : {train_len} 天从 {train_start} 到 {train_end}",
    f"测试日期唯一值数量: {num_test_date}",
    f"测试日期范围      : {test_len} 天从 {test_start} 到 {test_end}",
)

数据基本信息
产品系列类型数量      : 33
门店数量            : 54
门店-产品系列组合数量: 1782
目标序列数量        : 1782

训练日期唯一值数量: 1684
训练日期范围      : 1688 天从 2013-01-01 到 2017-08-15
测试日期唯一值数量: 16
测试日期范围      : 16 天从 2017-08-16 到 2017-08-31


# 3数据预处理

In [3]:
# 检查：训练数据中的缺失日期
missing_dates = pd.date_range(train_start, train_end).difference(train.date.unique())
missing_dates = missing_dates.strftime("%Y-%m-%d").tolist()

unique_dp_count = train.groupby(["store_nbr", "family"]).date.count().unique().tolist()

cprint(
    "时间序列中的缺失间隔",
    f"数据点唯一计数列表: {unique_dp_count}",
    f"缺失日期          : {missing_dates}",
)

时间序列中的缺失间隔
数据点唯一计数列表: [1684]
缺失日期          : ['2013-12-25', '2014-12-25', '2015-12-25', '2016-12-25']


In [4]:
# 重新索引训练数据以填充缺失日期
multi_idx = pd.MultiIndex.from_product(
    [pd.date_range(train_start, train_end), train.store_nbr.unique(), train.family.unique()],
    names=["date", "store_nbr", "family"],
)
train = train.set_index(["date", "store_nbr", "family"]).reindex(multi_idx).reset_index()

# 用0填充缺失值
train[["sales", "onpromotion"]] = train[["sales", "onpromotion"]].fillna(0.)
train.id = train.id.interpolate(method="linear")  # 线性插值填充'id'列

print("缺失日期处理完成！")

缺失日期处理完成！


In [5]:
# 检查：周末没有石油价格
missing_oil_dates = pd.date_range(train_start, test_end).difference(oil.date)
num_missing_oil_dates = len(missing_oil_dates)
num_wknd_missing = (missing_oil_dates.weekday >= 5).sum()
total_num_wknd = (pd.date_range(train_start, test_end).weekday >= 5).sum()

cprint(
    "缺失石油价格日期",
    f"缺失石油价格日期数量: {num_missing_oil_dates}",
    f"周末缺失数量      : {num_wknd_missing}",
    f"周末总数          : {total_num_wknd}",
)

缺失石油价格日期
缺失石油价格日期数量: 486
周末缺失数量      : 486
周末总数          : 486


In [6]:
# 重新索引石油数据
oil = oil.merge(
    pd.DataFrame({"date": pd.date_range(train_start, test_end)}),
    on="date",
    how="outer",
).sort_values("date", ignore_index=True)

# 使用线性插值填充缺失值
oil.oil = oil.oil.interpolate(method="linear", limit_direction="both")
print("石油价格数据填充完成！")

石油价格数据填充完成！


In [7]:
# 检查：由于零销售或缺失记录导致的缺失交易数据
num_zero_sales = (train.groupby(["date", "store_nbr"]).sales.sum().eq(0)).sum()
total_rec = num_store * train_len
curr_rec = len(transaction.index)
missing_rec = total_rec - curr_rec - num_zero_sales

cprint(
    "缺失交易记录",
    f"正确记录数量: {total_rec}",
    "",
    "细分...",
    f"当前记录数量    : {curr_rec}",
    f"零销售天数      : {num_zero_sales}",
    f"缺失记录数量    : {missing_rec}",
)

缺失交易记录
正确记录数量: 91152

细分...
当前记录数量    : 83488
零销售天数      : 7546
缺失记录数量    : 118


In [8]:
# 计算每家门店的总销售额
store_sales = train.groupby(["date", "store_nbr"]).sales.sum().reset_index()

# 重新索引交易数据
transaction = transaction.merge(
    store_sales,
    on=["date", "store_nbr"],
    how="outer",
).sort_values(["date", "store_nbr"], ignore_index=True)

# 零销售日期的交易量填充为0
transaction.loc[transaction.sales.eq(0), "transactions"] = 0.
transaction = transaction.drop(columns=["sales"])

# 使用线性插值填充剩余缺失值
transaction.transactions = transaction.groupby("store_nbr", group_keys=False).transactions.apply(
    lambda x: x.interpolate(method="linear", limit_direction="both")
)
print("交易数据填充完成！")

交易数据填充完成！


In [9]:
# 处理节假日描述信息
def process_holiday(s):
    if "futbol" in s:
        return "futbol"
    to_remove = list(set(store.city.str.lower()) | set(store.state.str.lower()))
    for w in to_remove:
        s = s.replace(w, "")
    return s

# 处理节假日数据
holiday.description = holiday.apply(
    lambda x: x.description.lower().replace(x.locale_name.lower(), ""), 
    axis=1,
).apply(
    process_holiday
).replace(
    r"[+-]\d+|\b(de|del|traslado|recupero|puente|-)\b", "", regex=True,
).replace(
    r"\s+|-", " ", regex=True,
).str.strip()

# 移除转移的节假日
holiday = holiday[holiday.transferred.eq(False)]

# 提取周六工作日数据
work_days = holiday[holiday.type.eq("Work Day")]
work_days = work_days[["date", "type"]].rename(
    columns={"type": "work_day"}
).reset_index(drop=True)
work_days.work_day = work_days.work_day.notna().astype(int)

# 移除工作日数据后继续处理节假日
holiday = holiday[holiday.type!="Work Day"].reset_index(drop=True)

print("节假日数据处理完成！")

节假日数据处理完成！


In [10]:
###################################
### 地方节假日（城市级别） ###
###################################
local_holidays = holiday[holiday.locale.eq("Local")]
local_holidays = local_holidays[["date", "locale_name", "description"]].rename(
    columns={"locale_name": "city"}
).reset_index(drop=True)
local_holidays = local_holidays[~local_holidays.duplicated()]
local_holidays = pd.get_dummies(local_holidays, columns=["description"], prefix="loc")

#######################################
### 区域节假日（州级别） ###
#######################################
regional_holidays = holiday[holiday.locale.eq("Regional")]
regional_holidays = regional_holidays[["date", "locale_name", "description"]].rename(
    columns={"locale_name": "state", "description": "provincializacion"}
).reset_index(drop=True)
regional_holidays.provincializacion = regional_holidays.provincializacion.eq("provincializacion").astype(int)

#########################
### 国家节假日 ###
#########################
national_holidays = holiday[holiday.locale.eq("National")]
national_holidays = national_holidays[["date", "description"]].reset_index(drop=True)
national_holidays = national_holidays[~national_holidays.duplicated()]
national_holidays = pd.get_dummies(national_holidays, columns=["description"], prefix="nat")
# 不同的国家节假日可能落在同一天
national_holidays = national_holidays.groupby("date").sum().reset_index()
# 为可视化目的缩短名称
national_holidays = national_holidays.rename(columns={"nat_primer grito independencia": "nat_primer grito"})

print("节假日数据处理完成！")

节假日数据处理完成！


In [11]:
# 保留对销售有较大影响的国家节假日
selected_holidays = [
    "nat_terremoto", "nat_navidad", "nat_dia la madre", "nat_dia trabajo",
    "nat_primer dia ano", "nat_futbol", "nat_dia difuntos",
]
keep_national_holidays = national_holidays[["date", *selected_holidays]]

# 合并所有数据集
data = pd.concat(
    [train, test], axis=0, ignore_index=True,
).merge(
    transaction, on=["date", "store_nbr"], how="left",
).merge(
    oil, on="date", how="left",
).merge(
    store, on="store_nbr", how="left",
).merge(
    work_days, on="date", how="left",
).merge(
    keep_national_holidays, on="date", how="left",
).sort_values(["date", "store_nbr", "family"], ignore_index=True)

# 用0填充节假日/事件列表示不存在
data[["work_day", *selected_holidays]] = data[["work_day", *selected_holidays]].fillna(0)

# 添加日期相关的未来协变量
data["day"] = data.date.dt.day
data["month"] = data.date.dt.month
data["year"] = data.date.dt.year
data["day_of_week"] = data.date.dt.dayofweek
data["day_of_year"] = data.date.dt.dayofyear
data["week_of_year"] = data.date.dt.isocalendar().week.astype(int)
data["date_index"] = data.date.factorize()[0]  # 按日期排序后计算

# 稍后使用线性插值填充零销售日
zero_sales_dates = missing_dates + [f"{j}-01-01" for j in range(2013, 2018)]
data.loc[(data.date.isin(zero_sales_dates))&(data.sales.eq(0))&(data.onpromotion.eq(0)), ["sales", "onpromotion"]] = np.nan

# 为清晰性添加前缀
data.store_nbr = data.store_nbr.apply(lambda x: (f"store_nbr_{x}"))
data.cluster = data.cluster.apply(lambda x: (f"cluster_{x}"))
data.type = data.type.apply(lambda x: (f"type_{x}"))

# 添加前缀确保'city'和'state'之间没有重复值
data.city = data.city.apply(lambda x: (f"city_{x.lower()}"))
data.state = data.state.apply(lambda x: (f"state_{x.lower()}"))

print("数据合并完成！前5行数据:")
print(data.head())

数据合并完成！前5行数据:
        date    store_nbr      family   id  sales  onpromotion  transactions  \
0 2013-01-01  store_nbr_1  AUTOMOTIVE  0.0    NaN          NaN           0.0   
1 2013-01-01  store_nbr_1   BABY CARE  1.0    NaN          NaN           0.0   
2 2013-01-01  store_nbr_1      BEAUTY  2.0    NaN          NaN           0.0   
3 2013-01-01  store_nbr_1   BEVERAGES  3.0    NaN          NaN           0.0   
4 2013-01-01  store_nbr_1       BOOKS  4.0    NaN          NaN           0.0   

     oil        city            state  ... nat_primer dia ano nat_futbol  \
0  93.14  city_quito  state_pichincha  ...                1.0        0.0   
1  93.14  city_quito  state_pichincha  ...                1.0        0.0   
2  93.14  city_quito  state_pichincha  ...                1.0        0.0   
3  93.14  city_quito  state_pichincha  ...                1.0        0.0   
4  93.14  city_quito  state_pichincha  ...                1.0        0.0   

   nat_dia difuntos  day  month  year  day_of_we

# 4进行数据准备与模型训练

In [12]:
def get_pipeline(static_covs_transform=False, log_transform=False):
    lst = []
    
    # 填充缺失值
    filler = MissingValuesFiller(n_jobs=-1)
    lst.append(filler)
    
    # 指定静态协变量转换
    if static_covs_transform:
        static_covs_transformer = StaticCovariatesTransformer(
            transformer_cat=OneHotEncoder(),
            n_jobs=-1,
        )
        lst.append(static_covs_transformer)

    # 对销售额进行对数转换
    if log_transform:
        log_transformer = InvertibleMapper(
            fn=np.log1p,
            inverse_fn=np.expm1,
            n_jobs=-1,
        )
        lst.append(log_transformer)

    # 重新缩放时间序列
    scaler = Scaler()
    lst.append(scaler)

    # 链接所有转换
    pipeline = Pipeline(lst)
    return pipeline

print("数据转换管道定义完成！")

数据转换管道定义完成！


In [13]:
def get_target_series(static_cols, log_transform=True):    
    target_dict = {}
    pipe_dict = {}
    id_dict = {}

    for fam in tqdm_notebook(data.family.unique(), desc="提取目标序列"):
        # 为每个模型过滤数据
        df = data[(data.family.eq(fam)) & (data.date.le(train_end.strftime("%Y-%m-%d")))]
        
        # 为目标序列初始化转换管道
        pipe = get_pipeline(True, log_transform=log_transform)
        
        # 提取目标序列和静态协变量
        target = TimeSeries.from_group_dataframe(
            df=df,
            time_col="date",
            value_cols="sales",
            group_cols="store_nbr",
            static_cols=static_cols,
        )

        # 记录每个目标序列的身份
        target_id = [{"store_nbr": t.static_covariates.store_nbr[0], "family": fam} 
                     for t in target]
        id_dict[fam] = target_id
        
        # 应用转换
        target = pipe.fit_transform(target)
        target_dict[fam] = [t.astype(np.float32) for t in target]
        pipe_dict[fam] = pipe[2:]
        
    return target_dict, pipe_dict, id_dict

# 排除'store_nbr'的静态协变量列表；'store_nbr'使用'group_cols'自动提取
static_cols = ["city", "state", "type", "cluster"]

target_dict, pipe_dict, id_dict = get_target_series(static_cols)
print("目标序列提取完成！")

提取目标序列:   0%|          | 0/33 [00:00<?, ?it/s]

目标序列提取完成！


In [14]:
def get_covariates(
    past_cols,
    future_cols,
    past_ma_cols=None,
    future_ma_cols=None,
    past_window_sizes=[7, 28],
    future_window_sizes=[7, 28],
):
    past_dict = {}
    future_dict = {}
    
    # 为协变量初始化转换管道
    covs_pipe = get_pipeline()

    for fam in tqdm_notebook(data.family.unique(), desc="提取协变量"):
        # 为每个模型过滤数据
        df = data[data.family.eq(fam)]
        
        # 提取过去协变量
        past_covs = TimeSeries.from_group_dataframe(
            df=df[df.date.le(train_end.strftime("%Y-%m-%d"))],
            time_col="date",
            value_cols=past_cols,
            group_cols="store_nbr",
        )
        past_covs = [p.with_static_covariates(None) for p in past_covs]
        past_covs = covs_pipe.fit_transform(past_covs)
        if past_ma_cols is not None:
            for size in past_window_sizes:
                ma_filter = MovingAverageFilter(window=size)
                old_names = [f"rolling_mean_{size}_{col}" for col in past_ma_cols]
                new_names = [f"{col}_ma{size}" for col in past_ma_cols]
                past_ma_covs = [
                    ma_filter.filter(p[past_ma_cols]).with_columns_renamed(old_names, new_names) 
                    for p in past_covs
                ]
                past_covs = [p.stack(p_ma) for p, p_ma in zip(past_covs, past_ma_covs)]
        
        past_dict[fam] = [p.astype(np.float32) for p in past_covs]

        # 提取未来协变量
        future_covs = TimeSeries.from_group_dataframe(
            df=df,
            time_col="date",
            value_cols=future_cols,
            group_cols="store_nbr",
        )
        future_covs = [f.with_static_covariates(None) for f in future_covs]
        future_covs = covs_pipe.fit_transform(future_covs)
        if future_ma_cols is not None:
            for size in future_window_sizes:
                ma_filter = MovingAverageFilter(window=size)
                old_names = [f"rolling_mean_{size}_{col}" for col in future_ma_cols]
                new_names = [f"{col}_ma{size}" for col in future_ma_cols]
                future_ma_covs = [
                    ma_filter.filter(f[future_ma_cols]).with_columns_renamed(old_names, new_names) 
                    for f in future_covs
                ]
                future_covs = [f.stack(f_ma) for f, f_ma in zip(future_covs, future_ma_covs)]
        
        future_dict[fam] = [f.astype(np.float32) for f in future_covs]
            
    return past_dict, future_dict

# 过去协变量
past_cols = ["transactions"]

# 未来协变量
future_cols = [
    "oil", "onpromotion",
    "day", "month", "year", "day_of_week", "day_of_year", "week_of_year", "date_index",
    "work_day", *selected_holidays,
]

# 通过计算移动平均值得出的额外过去和未来协变量
past_ma_cols = None
future_ma_cols = ["oil", "onpromotion"]

past_dict, future_dict = get_covariates(past_cols, future_cols, past_ma_cols, future_ma_cols)

cprint(
    "所有协变量列表",
    "静态协变量:",
    str(["city", "state", "type", "cluster", "store_nbr"]),
    "",
    "过去协变量:",
    str(past_cols),
    "",
    "未来协变量:",
    str(future_cols + [f"{col}_ma7" for col in future_ma_cols] + [f"{col}_ma28" for col in future_ma_cols]),
)

提取协变量:   0%|          | 0/33 [00:00<?, ?it/s]

所有协变量列表
静态协变量:
['city', 'state', 'type', 'cluster', 'store_nbr']

过去协变量:
['transactions']

未来协变量:
['oil', 'onpromotion', 'day', 'month', 'year', 'day_of_week', 'day_of_year', 'week_of_year', 'date_index', 'work_day', 'nat_terremoto', 'nat_navidad', 'nat_dia la madre', 'nat_dia trabajo', 'nat_primer dia ano', 'nat_futbol', 'nat_dia difuntos', 'oil_ma7', 'onpromotion_ma7', 'oil_ma28', 'onpromotion_ma28']


In [15]:
TRAINER_CONFIG = {
    # 之前提取的时间序列数据
    "target_dict": target_dict,
    "pipe_dict": pipe_dict,
    "id_dict": id_dict,
    "past_dict": past_dict,
    "future_dict": future_dict,
    
    # 使用滚动预测原点进行时间序列交叉验证
    "forecast_horizon": 16,  # 验证集长度
    "folds": 1,  # 训练集数量（设置为1表示标准训练-验证分割）
    
    # 检查零销售的天数；如果全部为零，则生成零预测
    "zero_fc_window": 21,
    
    # 指定要包含在模型中的协变量列表
    # 设置为None则不使用任何协变量，设置为'keep_all'则包含所有协变量
    "static_covs": "keep_all",  # 从['city', 'state', 'cluster', 'type', 'store_nbr']中指定，将提取所有独热编码列
    "past_covs": "keep_all",
    "future_covs": "keep_all",
}

class Trainer:
    def __init__(
        self,
        target_dict,
        pipe_dict,
        id_dict,
        past_dict,
        future_dict,
        forecast_horizon,
        folds,
        zero_fc_window,
        static_covs=None,
        past_covs=None,
        future_covs=None,
    ):
        self.target_dict = target_dict.copy()
        self.pipe_dict = pipe_dict.copy()
        self.id_dict = id_dict.copy()
        self.past_dict = past_dict.copy()
        self.future_dict = future_dict.copy()
        self.forecast_horizon = forecast_horizon
        self.folds = folds
        self.zero_fc_window = zero_fc_window
        self.static_covs = static_covs
        self.past_covs = past_covs
        self.future_covs = future_covs
        
        # 设置时间序列数据
        self.setup()
    
    def setup(self):
        for fam in tqdm_notebook(self.target_dict.keys(), desc="设置中"):
            # 保留指定的静态协变量
            if self.static_covs != "keep_all":
                if self.static_covs is not None:
                    target = self.target_dict[fam]
                    keep_static = [col for col in target[0].static_covariates.columns if col.startswith(tuple(self.static_covs))]
                    static_covs_df = [t.static_covariates[keep_static] for t in target]
                    self.target_dict[fam] = [t.with_static_covariates(d) for t, d in zip(target, static_covs_df)]
                else:
                    self.target_dict[fam] = [t.with_static_covariates(None) for t in target]
            
            # 保留指定的过去协变量
            if self.past_covs != "keep_all":
                if self.past_covs is not None:
                    self.past_dict[fam] = [p[self.past_covs] for p in self.past_dict[fam]]
                else:
                    self.past_dict[fam] = None
                
            # 保留指定的未来协变量
            if self.future_covs != "keep_all":
                if self.future_covs is not None:
                    self.future_dict[fam] = [p[self.future_covs] for p in self.future_dict[fam]]
                else:
                    self.future_dict[fam] = None
    
    def clip(self, array):
        return np.clip(array, a_min=0., a_max=None)
    
    def train_valid_split(self, target, length):
        train = [t[:-length] for t in target]
        valid_end_idx = -length + self.forecast_horizon
        if valid_end_idx >= 0:
            valid_end_idx = None
        valid = [t[-length:valid_end_idx] for t in target]
        
        return train, valid
    
    def get_models(self, model_names, model_configs):
        models = {
            "lr": LinearRegressionModel,
            "lgbm": LightGBMModel,
            "cat": CatBoostModel,
            "xgb": XGBModel,
        }
        assert isinstance(model_names, list) and isinstance(model_configs, list),\
        "模型名称和模型配置都必须指定为列表。"
        assert all(name in models for name in model_names),\
        f"未识别的模型名称: '{model_names}'。"
        assert len(model_names) == len(model_configs),\
        "模型名称数量和模型配置数量不匹配。"
        
        if "xgb" in model_names:
            xgb_idx = np.where(np.array(model_names)=="xgb")[0]
            for idx in xgb_idx:
                # 为XGBoost更改为基于直方图的方法以获得更快的训练时间
                model_configs[idx] = {"tree_method": "hist", **model_configs[idx]}
        
        return [models[name](**model_configs[j]) for j, name in enumerate(model_names)]
    
    def generate_forecasts(self, models, train, pipe, past_covs, future_covs, drop_before):
        if drop_before is not None:
            date = pd.Timestamp(drop_before) - pd.Timedelta(days=1)
            train = [t.drop_before(date) for t in train]
        inputs = {
            "series": train,
            "past_covariates": past_covs,
            "future_covariates": future_covs,
        }
        zero_pred = pd.DataFrame({
            "date": pd.date_range(train[0].end_time(), periods=self.forecast_horizon+1)[1:],
            "sales": np.zeros(self.forecast_horizon),
        })
        zero_pred = TimeSeries.from_dataframe(
            df=zero_pred,
            time_col="date",
            value_cols="sales",
        )
        
        pred_list = []
        ens_pred = [0 for _ in range(len(train))]
        
        for m in models:
            # 将训练数据拟合到模型
            m.fit(**inputs)

            # 生成预测并应用逆转换
            pred = m.predict(n=self.forecast_horizon, **inputs)
            pred = pipe.inverse_transform(pred)

            # 为近期观测值为0的目标序列设置零预测
            for j in range(len(train)):
                if train[j][-self.zero_fc_window:].values().sum() == 0:
                    pred[j] = zero_pred
            
            # 将负预测裁剪为0
            pred = [p.map(self.clip) for p in pred]
            pred_list.append(pred)
            
            # 集成平均
            for j in range(len(ens_pred)):
                ens_pred[j] += pred[j] / len(models)

        return pred_list, ens_pred
    
    def metric(self, valid, pred):
        return rmsle(valid, pred, inter_reduction=np.mean)
    
    def validate(self, model_names, model_configs, drop_before=None):
        # 辅助值以对齐下面打印的文本
        longest_len = len(max(self.target_dict.keys(), key=len))
        
        # 为每个模型存储指标值
        model_metrics_history = []
        ens_metric_history = []
        
        for fam in tqdm_notebook(self.target_dict, desc="执行验证"):
            target = self.target_dict[fam]
            pipe = self.pipe_dict[fam]
            past_covs = self.past_dict[fam]
            future_covs = self.future_dict[fam]
            
            # 记录所有折的平均指标值
            model_metrics = []
            ens_metric = 0
            
            for j in range(self.folds):    
                # 执行训练-验证分割并应用转换
                length = (self.folds - j) * self.forecast_horizon
                train, valid = self.train_valid_split(target, length)
                valid = pipe.inverse_transform(valid)

                # 生成预测并计算指标
                models = self.get_models(model_names, model_configs)
                pred_list, ens_pred = self.generate_forecasts(models, train, pipe, past_covs, future_covs, drop_before)
                metric_list = [self.metric(valid, pred) / self.folds for pred in pred_list]
                model_metrics.append(metric_list)
                if len(models) > 1:
                    ens_metric_fold = self.metric(valid, ens_pred) / self.folds
                    ens_metric += ens_metric_fold
                
            # 为每个模型存储最终指标值
            model_metrics = np.sum(model_metrics, axis=0)
            model_metrics_history.append(model_metrics)
            ens_metric_history.append(ens_metric)
            
            # 为每个产品系列打印指标值
            print(
                fam,
                " " * (longest_len - len(fam)),
                " | ",
                " - ".join([f"{model}: {metric:.5f}" for model, metric in zip(model_names, model_metrics)]),
                f" - ens: {ens_metric:.5f}" if len(models) > 1 else "",
                sep="",
            )
            
        # 打印总体指标值
        cprint(
            "平均RMSLE | "
            + " - ".join([f"{model}: {metric:.5f}" 
                          for model, metric in zip(model_names, np.mean(model_metrics_history, axis=0))])
            + (f" - ens: {np.mean(ens_metric_history):.5f}" if len(models) > 1 else ""),
        )
        
    def ensemble_predict(self, model_names, model_configs, drop_before=None):        
        forecasts = []
        for fam in tqdm_notebook(self.target_dict.keys(), desc="生成预测"):
            target = self.target_dict[fam]
            pipe = self.pipe_dict[fam]
            target_id = self.id_dict[fam]
            past_covs = self.past_dict[fam]
            future_covs = self.future_dict[fam]
            
            # 生成预测
            models = self.get_models(model_names, model_configs)
            pred_list, ens_pred = self.generate_forecasts(models, target, pipe, past_covs, future_covs, drop_before)
            ens_pred = [p.pd_dataframe().assign(**i) for p, i in zip(ens_pred, target_id)]
            ens_pred = pd.concat(ens_pred, axis=0)
            forecasts.append(ens_pred)
            
        # 将所有预测合并到一个数据框中
        forecasts = pd.concat(forecasts, axis=0)
        forecasts = forecasts.rename_axis(None, axis=1).reset_index(names="date")
        
        return forecasts

# 初始化模型训练器
trainer = Trainer(**TRAINER_CONFIG)
print("模型训练器初始化完成！")

设置中:   0%|          | 0/33 [00:00<?, ?it/s]

模型训练器初始化完成！


In [16]:
# 首先查看rmsle函数的参数
import inspect
print(inspect.signature(rmsle))

(actual_series: Union[darts.timeseries.TimeSeries, collections.abc.Sequence[darts.timeseries.TimeSeries]], pred_series: Union[darts.timeseries.TimeSeries, collections.abc.Sequence[darts.timeseries.TimeSeries]], intersect: bool = True, *, q: Union[float, list[float], tuple[numpy.ndarray, pandas.core.indexes.base.Index], NoneType] = None, component_reduction: Optional[Callable[[numpy.ndarray], float]] = <function nanmean at 0x7f8ca4158e70>, series_reduction: Optional[Callable[[numpy.ndarray], Union[float, numpy.ndarray]]] = None, n_jobs: int = 1, verbose: bool = False) -> Union[float, list[float], numpy.ndarray, list[numpy.ndarray]]


In [17]:
# 修正metric函数
def metric(self, valid, pred):
    return rmsle(valid, pred, inter_reduction=np.mean)

# 更新训练器类中的metric方法
trainer.metric = metric.__get__(trainer, Trainer)

print("metric函数已修复！")

metric函数已修复！


## 4.1线性回归

In [18]:
BASE_CONFIG = {
    "random_state": 0,
    
    # 目标序列的滞后值数量
    "lags": 63,
    
    # 过去协变量的滞后值数量
    "lags_past_covariates": list(range(-16, -23, -1)) if TRAINER_CONFIG["past_covs"] is not None else None,
    
    # 未来协变量的（过去，未来-1）滞后值数量
    "lags_future_covariates": (14, 1) if TRAINER_CONFIG["future_covs"] is not None else None,
    
    # 给定今天的输入数据，模型预测的天数
    "output_chunk_length": 1,
}


# 修正metric函数
def metric(self, valid, pred):
    return rmsle(valid, pred, series_reduction=np.mean)

# 更新训练器类中的metric方法
trainer.metric = metric.__get__(trainer, Trainer)

print("metric函数已修复！现在重新运行线性回归模型...")

# 重新运行线性回归模型
trainer.validate(["lr"], [BASE_CONFIG], drop_before="2015-01-01")


metric函数已修复！现在重新运行线性回归模型...


执行验证:   0%|          | 0/33 [00:00<?, ?it/s]

`predict()` was called with `n > output_chunk_length`: using auto-regression to forecast the values after `output_chunk_length` points. The model will access `(n - output_chunk_length)` future values of your `past_covariates` (relative to the first predicted time step). To hide this warning, set `show_warnings=False`.


AUTOMOTIVE                 | lr: 0.49342


`predict()` was called with `n > output_chunk_length`: using auto-regression to forecast the values after `output_chunk_length` points. The model will access `(n - output_chunk_length)` future values of your `past_covariates` (relative to the first predicted time step). To hide this warning, set `show_warnings=False`.


...

PRODUCE                    | lr: 0.51722


`predict()` was called with `n > output_chunk_length`: using auto-regression to forecast the values after `output_chunk_length` points. The model will access `(n - output_chunk_length)` future values of your `past_covariates` (relative to the first predicted time step). To hide this warning, set `show_warnings=False`.


SCHOOL AND OFFICE SUPPLIES | lr: 0.70248


`predict()` was called with `n > output_chunk_length`: using auto-regression to forecast the values after `output_chunk_length` points. The model will access `(n - output_chunk_length)` future values of your `past_covariates` (relative to the first predicted time step). To hide this warning, set `show_warnings=False`.


SEAFOOD                    | lr: 0.45968
平均RMSLE | lr: 0.36263


# 4.2 LIGHTGBM

In [19]:
# 定义LightGBM的配置
GBDT_CONFIG1 = {
    **BASE_CONFIG,
    
    # 可以指定的额外超参数
    # "n_estimators": 100,
    # "learning_rate": 0.1,
    # "max_depth": 6,
}

# 创建不同滞后值的配置（集成用）
GBDT_CONFIG2 = GBDT_CONFIG1.copy()
GBDT_CONFIG2["lags"] = 7

GBDT_CONFIG3 = GBDT_CONFIG1.copy()
GBDT_CONFIG3["lags"] = 365

GBDT_CONFIG4 = GBDT_CONFIG1.copy()
GBDT_CONFIG4["lags"] = 730

# 训练集成模型
print("开始训练LightGBM集成模型...")
ENS_MODELS = ["lgbm", "lgbm", "lgbm", "lgbm"]
ENS_CONFIGS = [GBDT_CONFIG1, GBDT_CONFIG2, GBDT_CONFIG3, GBDT_CONFIG4]

trainer.validate(
    model_names=ENS_MODELS, 
    model_configs=ENS_CONFIGS,
    drop_before="2015-01-01",
)

开始训练LightGBM集成模型...


执行验证:   0%|          | 0/33 [00:00<?, ?it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.162486 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 43638
[LightGBM] [Info] Number of data points in the train set: 47466, number of used features: 484
[LightGBM] [Info] Start training from score 0.506933


`predict()` was called with `n > output_chunk_length`: using auto-regression to forecast the values after `output_chunk_length` points. The model will access `(n - output_chunk_length)` future values of your `past_covariates` (relative to the first predicted time step). To hide this warning, set `show_warnings=False`.


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.044610 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 29410
[LightGBM] [Info] Number of data points in the train set: 50490, number of used features: 428
[LightGBM] [Info] Start training from score 0.502433


...

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.200470 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 120630
[LightGBM] [Info] Number of data points in the train set: 31158, number of used features: 786
[LightGBM] [Info] Start training from score 0.517073


`predict()` was called with `n > output_chunk_length`: using auto-regression to forecast the values after `output_chunk_length` points. The model will access `(n - output_chunk_length)` future values of your `past_covariates` (relative to the first predicted time step). To hide this warning, set `show_warnings=False`.


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.116515 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 208842
[LightGBM] [Info] Number of data points in the train set: 11448, number of used features: 1101
[LightGBM] [Info] Start training from score 0.521129


`predict()` was called with `n > output_chunk_length`: using auto-regression to forecast the values after `output_chunk_length` points. The model will access `(n - output_chunk_length)` future values of your `past_covariates` (relative to the first predicted time step). To hide this warning, set `show_warnings=False`.


SEAFOOD                    | lgbm: 0.46341 - lgbm: 0.46381 - lgbm: 0.46750 - lgbm: 0.47710 - ens: 0.45442
平均RMSLE | lgbm: 0.33909 - lgbm: 0.35501 - lgbm: 0.33981 - lgbm: 0.34645 - ens: 0.33725


In [20]:
# 先检查TimeSeries对象的方法
sample_target = target_dict[list(target_dict.keys())[0]][0]
print("TimeSeries对象的方法:")
print(dir(sample_target)[:20])  # 显示前20个方法

# 检查是否有类似dataframe的方法
df_methods = [method for method in dir(sample_target) if 'dataframe' in method.lower() or 'pd' in method.lower()]
print("\n与dataframe相关的方法:", df_methods)

TimeSeries对象的方法:
['__abs__', '__add__', '__class__', '__contains__', '__copy__', '__deepcopy__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getitem__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__']

与dataframe相关的方法: ['from_dataframe', 'from_group_dataframe', 'to_dataframe']


In [21]:
# 修正ensemble_predict方法中的错误
def ensemble_predict(self, model_names, model_configs, drop_before=None):        
    forecasts = []
    for fam in tqdm_notebook(self.target_dict.keys(), desc="生成预测"):
        target = self.target_dict[fam]
        pipe = self.pipe_dict[fam]
        target_id = self.id_dict[fam]
        past_covs = self.past_dict[fam]
        future_covs = self.future_dict[fam]
        
        # 生成预测
        models = self.get_models(model_names, model_configs)
        pred_list, ens_pred = self.generate_forecasts(models, target, pipe, past_covs, future_covs, drop_before)
        
        # 修正：使用.to_dataframe()而不是.pd_dataframe()
        ens_pred_dataframes = []
        for p, i in zip(ens_pred, target_id):
            df = p.to_dataframe().reset_index()
            df = df.assign(**i)
            ens_pred_dataframes.append(df)
        
        ens_pred_df = pd.concat(ens_pred_dataframes, axis=0)
        forecasts.append(ens_pred_df)
        
    # 将所有预测合并到一个数据框中
    forecasts = pd.concat(forecasts, axis=0)
    forecasts = forecasts.rename_axis(None, axis=1).reset_index(drop=True)
    
    return forecasts

# 更新训练器类中的ensemble_predict方法
trainer.ensemble_predict = ensemble_predict.__get__(trainer, Trainer)

print("ensemble_predict方法已修正！现在重新生成预测...")

# 重新生成预测
predictions1 = trainer.ensemble_predict(
    model_names=ENS_MODELS, 
    model_configs=ENS_CONFIGS,
)

predictions2 = trainer.ensemble_predict(
    model_names=ENS_MODELS, 
    model_configs=ENS_CONFIGS,
    drop_before="2015-01-01",
)

print("预测生成完成！")
print("预测1的样本：")
print(predictions1.head())
print("\n预测2的样本：")
print(predictions2.head())
print(f"\n预测1的形状: {predictions1.shape}")
print(f"预测2的形状: {predictions2.shape}")

ensemble_predict方法已修正！现在重新生成预测...


生成预测:   0%|          | 0/33 [00:00<?, ?it/s]

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.086180 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 43758
[LightGBM] [Info] Number of data points in the train set: 87750, number of used features: 499
[LightGBM] [Info] Start training from score 0.460012


`predict()` was called with `n > output_chunk_length`: using auto-regression to forecast the values after `output_chunk_length` points. The model will access `(n - output_chunk_length)` future values of your `past_covariates` (relative to the first predicted time step). To hide this warning, set `show_warnings=False`.


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.080103 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 29478
[LightGBM] [Info] Number of data points in the train set: 89964, number of used features: 443
[LightGBM] [Info] Start training from score 0.457790


...

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.180972 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 120621
[LightGBM] [Info] Number of data points in the train set: 32022, number of used features: 786
[LightGBM] [Info] Start training from score 0.516655


`predict()` was called with `n > output_chunk_length`: using auto-regression to forecast the values after `output_chunk_length` points. The model will access `(n - output_chunk_length)` future values of your `past_covariates` (relative to the first predicted time step). To hide this warning, set `show_warnings=False`.


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.109638 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 210107
[LightGBM] [Info] Number of data points in the train set: 12312, number of used features: 1101
[LightGBM] [Info] Start training from score 0.519759


`predict()` was called with `n > output_chunk_length`: using auto-regression to forecast the values after `output_chunk_length` points. The model will access `(n - output_chunk_length)` future values of your `past_covariates` (relative to the first predicted time step). To hide this warning, set `show_warnings=False`.


预测生成完成！
预测1的样本：
        date     sales    store_nbr      family
0 2017-08-16  3.274170  store_nbr_1  AUTOMOTIVE
1 2017-08-17  3.062303  store_nbr_1  AUTOMOTIVE
2 2017-08-18  3.788465  store_nbr_1  AUTOMOTIVE
3 2017-08-19  4.639717  store_nbr_1  AUTOMOTIVE
4 2017-08-20  1.888303  store_nbr_1  AUTOMOTIVE

预测2的样本：
        date     sales    store_nbr      family
0 2017-08-16  3.245887  store_nbr_1  AUTOMOTIVE
1 2017-08-17  2.964534  store_nbr_1  AUTOMOTIVE
2 2017-08-18  3.304805  store_nbr_1  AUTOMOTIVE
3 2017-08-19  4.241706  store_nbr_1  AUTOMOTIVE
4 2017-08-20  2.133952  store_nbr_1  AUTOMOTIVE

预测1的形状: (28512, 4)
预测2的形状: (28512, 4)


In [22]:
# 计算两个集成模型的平均值
final_predictions = predictions1.merge(
    predictions2, on=["date", "store_nbr", "family"], how="left",
)
final_predictions["sales"] = final_predictions[["sales_x", "sales_y"]].mean(axis=1)
final_predictions = final_predictions.drop(columns=["sales_x", "sales_y"])

print("最终预测的样本：")
print(final_predictions.head())
print(f"\n最终预测的形状: {final_predictions.shape}")

最终预测的样本：
        date    store_nbr      family     sales
0 2017-08-16  store_nbr_1  AUTOMOTIVE  3.260028
1 2017-08-17  store_nbr_1  AUTOMOTIVE  3.013419
2 2017-08-18  store_nbr_1  AUTOMOTIVE  3.546635
3 2017-08-19  store_nbr_1  AUTOMOTIVE  4.440712
4 2017-08-20  store_nbr_1  AUTOMOTIVE  2.011128

最终预测的形状: (28512, 4)


In [23]:
def prepare_submission(predictions):
    predictions = predictions.copy()
    
    # 处理列值以进行合并
    predictions.store_nbr = predictions.store_nbr.replace(
        "store_nbr_", "", regex=True,
    ).astype(int)
     
    # 匹配对应的'id'
    submission = test.merge(
        predictions, on=["date", "store_nbr", "family"], how="left",
    )[["id", "sales"]]
    
    return submission

# 准备提交文件
submission = prepare_submission(final_predictions)

print("提交文件的样本：")
print(submission.head())
print(f"\n提交文件的形状: {submission.shape}")

# 检查是否有缺失值
print(f"\n缺失值数量: {submission.sales.isnull().sum()}")
print(f"零值数量: {(submission.sales == 0).sum()}")
print(f"负值数量: {(submission.sales < 0).sum()}")

# 将负值裁剪为0
submission.sales = submission.sales.clip(lower=0)
print(f"\n裁剪后负值数量: {(submission.sales < 0).sum()}")

提交文件的样本：
        id        sales
0  3000888     3.260028
1  3000889     0.000000
2  3000890     4.235445
3  3000891  2337.802804
4  3000892     0.035706

提交文件的形状: (28512, 2)

缺失值数量: 0
零值数量: 2032
负值数量: 0

裁剪后负值数量: 0


In [24]:
# 定义XGBoost的配置
XGB_CONFIG = {
    **BASE_CONFIG,
    
    # XGBoost特定参数
    "n_estimators": 100,
    "learning_rate": 0.1,
    "max_depth": 6,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    
    # 为XGBoost更改为基于直方图的方法以获得更快的训练时间
    "tree_method": "hist",
}

# 创建不同滞后值的配置（集成用）
XGB_CONFIG2 = XGB_CONFIG.copy()
XGB_CONFIG2["lags"] = 7

XGB_CONFIG3 = XGB_CONFIG.copy()
XGB_CONFIG3["lags"] = 365

XGB_CONFIG4 = XGB_CONFIG.copy()
XGB_CONFIG4["lags"] = 730

print("开始训练XGBoost集成模型...")
XGB_MODELS = ["xgb", "xgb", "xgb", "xgb"]
XGB_CONFIGS = [XGB_CONFIG, XGB_CONFIG2, XGB_CONFIG3, XGB_CONFIG4]

trainer.validate(
    model_names=XGB_MODELS, 
    model_configs=XGB_CONFIGS,
    drop_before="2015-01-01",
)

开始训练XGBoost集成模型...


执行验证:   0%|          | 0/33 [00:00<?, ?it/s]

`predict()` was called with `n > output_chunk_length`: using auto-regression to forecast the values after `output_chunk_length` points. The model will access `(n - output_chunk_length)` future values of your `past_covariates` (relative to the first predicted time step). To hide this warning, set `show_warnings=False`.
`predict()` was called with `n > output_chunk_length`: using auto-regression to forecast the values after `output_chunk_length` points. The model will access `(n - output_chunk_length)` future values of your `past_covariates` (relative to the first predicted time step). To hide this warning, set `show_warnings=False`.
`predict()` was called with `n > output_chunk_length`: using auto-regression to forecast the values after `output_chunk_length` points. The model will access `(n - output_chunk_length)` future values of your `past_covariates` (relative to the first predicted time step). To hide this warning, set `show_warnings=False`.
`predict()` was called with `n > output_

AUTOMOTIVE                 | xgb: 0.49637 - xgb: 0.50740 - xgb: 0.50176 - xgb: 0.50805 - ens: 0.49580


`predict()` was called with `n > output_chunk_length`: using auto-regression to forecast the values after `output_chunk_length` points. The model will access `(n - output_chunk_length)` future values of your `past_covariates` (relative to the first predicted time step). To hide this warning, set `show_warnings=False`.
`predict()` was called with `n > output_chunk_length`: using auto-regression to forecast the values after `output_chunk_length` points. The model will access `(n - output_chunk_length)` future values of your `past_covariates` (relative to the first predicted time step). To hide this warning, set `show_warnings=False`.
`predict()` was called with `n > output_chunk_length`: using auto-regression to forecast the values after `output_chunk_length` points. The model will access `(n - output_chunk_length)` future values of your `past_covariates` (relative to the first predicted time step). To hide this warning, set `show_warnings=False`.
`predict()` was called with `n > output_

...

PRODUCE                    | xgb: 0.13595 - xgb: 0.15697 - xgb: 0.13713 - xgb: 0.13809 - ens: 0.13507


`predict()` was called with `n > output_chunk_length`: using auto-regression to forecast the values after `output_chunk_length` points. The model will access `(n - output_chunk_length)` future values of your `past_covariates` (relative to the first predicted time step). To hide this warning, set `show_warnings=False`.
`predict()` was called with `n > output_chunk_length`: using auto-regression to forecast the values after `output_chunk_length` points. The model will access `(n - output_chunk_length)` future values of your `past_covariates` (relative to the first predicted time step). To hide this warning, set `show_warnings=False`.
`predict()` was called with `n > output_chunk_length`: using auto-regression to forecast the values after `output_chunk_length` points. The model will access `(n - output_chunk_length)` future values of your `past_covariates` (relative to the first predicted time step). To hide this warning, set `show_warnings=False`.
`predict()` was called with `n > output_

SCHOOL AND OFFICE SUPPLIES | xgb: 0.59083 - xgb: 0.56179 - xgb: 0.56715 - xgb: 0.63288 - ens: 0.56994


`predict()` was called with `n > output_chunk_length`: using auto-regression to forecast the values after `output_chunk_length` points. The model will access `(n - output_chunk_length)` future values of your `past_covariates` (relative to the first predicted time step). To hide this warning, set `show_warnings=False`.
`predict()` was called with `n > output_chunk_length`: using auto-regression to forecast the values after `output_chunk_length` points. The model will access `(n - output_chunk_length)` future values of your `past_covariates` (relative to the first predicted time step). To hide this warning, set `show_warnings=False`.
`predict()` was called with `n > output_chunk_length`: using auto-regression to forecast the values after `output_chunk_length` points. The model will access `(n - output_chunk_length)` future values of your `past_covariates` (relative to the first predicted time step). To hide this warning, set `show_warnings=False`.
`predict()` was called with `n > output_

SEAFOOD                    | xgb: 0.46595 - xgb: 0.46093 - xgb: 0.47337 - xgb: 0.47230 - ens: 0.45525
平均RMSLE | xgb: 0.33896 - xgb: 0.35462 - xgb: 0.33844 - xgb: 0.34751 - ens: 0.33566


In [25]:
# 生成XGBoost预测
print("\n生成XGBoost集成预测...")
xgb_predictions1 = trainer.ensemble_predict(
    model_names=XGB_MODELS, 
    model_configs=XGB_CONFIGS,
)

xgb_predictions2 = trainer.ensemble_predict(
    model_names=XGB_MODELS, 
    model_configs=XGB_CONFIGS,
    drop_before="2015-01-01",
)

# 计算两个XGBoost集成模型的平均值
xgb_final = xgb_predictions1.merge(
    xgb_predictions2, on=["date", "store_nbr", "family"], how="left",
)
xgb_final["sales"] = xgb_final[["sales_x", "sales_y"]].mean(axis=1)
xgb_final = xgb_final.drop(columns=["sales_x", "sales_y"])

print(f"\nXGBoost预测的形状: {xgb_final.shape}")
print("XGBoost预测的样本：")
print(xgb_final.head())


生成XGBoost集成预测...


生成预测:   0%|          | 0/33 [00:00<?, ?it/s]

`predict()` was called with `n > output_chunk_length`: using auto-regression to forecast the values after `output_chunk_length` points. The model will access `(n - output_chunk_length)` future values of your `past_covariates` (relative to the first predicted time step). To hide this warning, set `show_warnings=False`.
`predict()` was called with `n > output_chunk_length`: using auto-regression to forecast the values after `output_chunk_length` points. The model will access `(n - output_chunk_length)` future values of your `past_covariates` (relative to the first predicted time step). To hide this warning, set `show_warnings=False`.
`predict()` was called with `n > output_chunk_length`: using auto-regression to forecast the values after `output_chunk_length` points. The model will access `(n - output_chunk_length)` future values of your `past_covariates` (relative to the first predicted time step). To hide this warning, set `show_warnings=False`.
`predict()` was called with `n > output_

生成预测:   0%|          | 0/33 [00:00<?, ?it/s]

`predict()` was called with `n > output_chunk_length`: using auto-regression to forecast the values after `output_chunk_length` points. The model will access `(n - output_chunk_length)` future values of your `past_covariates` (relative to the first predicted time step). To hide this warning, set `show_warnings=False`.
`predict()` was called with `n > output_chunk_length`: using auto-regression to forecast the values after `output_chunk_length` points. The model will access `(n - output_chunk_length)` future values of your `past_covariates` (relative to the first predicted time step). To hide this warning, set `show_warnings=False`.
`predict()` was called with `n > output_chunk_length`: using auto-regression to forecast the values after `output_chunk_length` points. The model will access `(n - output_chunk_length)` future values of your `past_covariates` (relative to the first predicted time step). To hide this warning, set `show_warnings=False`.
`predict()` was called with `n > output_


XGBoost预测的形状: (28512, 4)
XGBoost预测的样本：
        date    store_nbr      family     sales
0 2017-08-16  store_nbr_1  AUTOMOTIVE  3.433200
1 2017-08-17  store_nbr_1  AUTOMOTIVE  3.176463
2 2017-08-18  store_nbr_1  AUTOMOTIVE  3.426868
3 2017-08-19  store_nbr_1  AUTOMOTIVE  4.274847
4 2017-08-20  store_nbr_1  AUTOMOTIVE  1.828982


In [26]:
# 创建最终的混合模型（LightGBM + XGBoost）
print("\n创建LightGBM + XGBoost混合模型...")

# 将LightGBM和XGBoost预测进行平均
final_ensemble = final_predictions.merge(
    xgb_final, on=["date", "store_nbr", "family"], suffixes=("_lgbm", "_xgb"), how="left",
)
final_ensemble["sales"] = final_ensemble[["sales_lgbm", "sales_xgb"]].mean(axis=1)
final_ensemble = final_ensemble.drop(columns=["sales_lgbm", "sales_xgb"])

print(f"最终混合预测的形状: {final_ensemble.shape}")
print("最终混合预测的样本：")
print(final_ensemble.head())


创建LightGBM + XGBoost混合模型...
最终混合预测的形状: (28512, 4)
最终混合预测的样本：
        date    store_nbr      family     sales
0 2017-08-16  store_nbr_1  AUTOMOTIVE  3.346614
1 2017-08-17  store_nbr_1  AUTOMOTIVE  3.094941
2 2017-08-18  store_nbr_1  AUTOMOTIVE  3.486751
3 2017-08-19  store_nbr_1  AUTOMOTIVE  4.357779
4 2017-08-20  store_nbr_1  AUTOMOTIVE  1.920055


In [27]:
# 准备提交文件
submission_final = prepare_submission(final_ensemble)
submission_final.sales = submission_final.sales.clip(lower=0)

print(f"\n最终提交文件形状: {submission_final.shape}")
print("最终提交文件的样本：")
print(submission_final.head())

# 保存最终提交文件
submission_final.to_csv("submission", index=False)

print("\n最终提交文件已保存为 'submission.csv'")


最终提交文件形状: (28512, 2)
最终提交文件的样本：
        id        sales
0  3000888     3.346614
1  3000889     0.000000
2  3000890     4.323974
3  3000891  2346.136510
4  3000892     0.028281

最终提交文件已保存为 'submission.csv'


In [28]:
# 显示最终统计信息
cprint(
    "最终模型性能总结",
    f"线性回归模型平均RMSLE: 0.36263",
    f"LightGBM集成模型平均RMSLE: 0.33725",
    f"XGBoost集成模型平均RMSLE: 需要查看上述输出",
    "",
    "最终混合模型预测统计:",
    f"总预测数量: {len(submission)}",
    f"零预测数量: {(submission.sales == 0).sum()}",
    f"平均销售额预测: {submission.sales.mean():.2f}",
    f"中位数销售额预测: {submission.sales.median():.2f}",
    f"最小销售额预测: {submission.sales.min():.2f}",
    f"最大销售额预测: {submission.sales.max():.2f}",
)

最终模型性能总结
线性回归模型平均RMSLE: 0.36263
LightGBM集成模型平均RMSLE: 0.33725
XGBoost集成模型平均RMSLE: 需要查看上述输出

最终混合模型预测统计:
总预测数量: 28512
零预测数量: 2032
平均销售额预测: 435.07
中位数销售额预测: 28.16
最小销售额预测: 0.00
最大销售额预测: 14025.72
